# 🧱 Property Calculator
**Version 2026-11-24 19:25** - Step 2 shows the whole materials table and a list to pick a material from (step 3 uses the pick); your Google Sheet 'materials' is the default table, with an option to leave out the generic values

Material properties and layer values for the **Thermal Matrix Model**. Look up a material, then
get the conductance UA and capacitance C of a layer, split into segments for thick, massive
layers, with formulas you can paste straight into the network editor.

**Your own materials table.** The built-in table holds generic typical values. To use
**ASHRAE Handbook—Fundamentals, chapter 26, Table 1** (*Thermal Properties of Common Building and
Insulating Materials*) or manufacturers' data, put the rows you need in a CSV or Google Sheet
with these columns (SI, as in Table 1):

| column | meaning |
|---|---|
| `material` | your name for it, e.g. *hollow clay block 25 cm* |
| `density_kg_m3` | density ρ, kg/m³ |
| `k_W_mK` | conductivity k, W/(m·K) |
| `R_m2K_W` and `R_thickness_mm` | for products listed by **resistance at a given thickness** (boards, blocks), give R and that thickness instead of k; k is worked out as thickness / R |
| `cp_kJ_kgK` | specific heat, kJ/(kg·K) |
| `source` | where the numbers came from (e.g. *ASHRAE Fund. 2025 ch. 26 Table 1*) |

**The easy way:** tick *save_template* in step 1 and run it. In Google Drive, open
*ThermalMatrixModel ▸ materials.csv* with **Google Sheets**. That makes a Google Sheet called
*materials*; add your rows there (Sheets saves as you type). Then choose **My Google Sheet** in step 1
and run it again (Colab asks once for permission to read your Sheets). There's no need to save as CSV.
That Sheet is the default source in step 1, so whatever is in it is used. To bring in a fuller
table (e.g. a class table of Table 1 values), paste it into the Sheet, or in Sheets use
*File ▸ Import ▸ Upload* with *Replace spreadsheet*, keeping the same columns. Your rows are
added to the generic ones (a row with the same name replaces the generic value); untick
*include_generic* to use your table alone. ASHRAE's handbook is copyrighted, so keep your table in your own
Drive (or the course site) rather than posting it publicly. Penn users can read the handbook
online through the library.

In [ ]:
#@title 0 ▸ Setup (run once) { display-mode: "form" }
"""
property_calculator.py -- material properties and layer values for the thermal matrix model.

A materials table has one row per material, laid out like Table 1 of the ASHRAE Handbook--
Fundamentals, chapter 26 (SI): density, conductivity k (or, for products sold at one
thickness, a resistance R at a stated thickness), and specific heat. The built-in table
holds generic typical values; students add rows from the handbook or product data in their
own CSV (see TEMPLATE_COLUMNS), which is merged on top.

For a layer of a given area and thickness it gives the values the network needs
(conduction UA, capacitance C), split into segments for thick massive layers, plus formulas
ready to paste into the network editor.
"""
import difflib
import io
import math
import os
import urllib.request

import numpy as np
import pandas as pd

CALC_VERSION = "2026-11-24 19:25"

# Columns of a materials CSV (SI, as in ASHRAE Fundamentals ch. 26, Table 1).
# Give k, or R with the thickness it applies to (k is then thickness / R).
TEMPLATE_COLUMNS = ["material", "density_kg_m3", "k_W_mK", "R_m2K_W", "R_thickness_mm",
                    "cp_kJ_kgK", "source"]

# Generic typical values (same as the thermal matrix model's MATERIALS library).
GENERIC = [
    ("concrete", 2200, 1.75, None, None, 0.92),
    ("brick", 1920, 0.72, None, None, 0.84),
    ("adobe", 1600, 0.52, None, None, 0.90),
    ("rammed earth", 2000, 1.10, None, None, 0.90),
    ("stone", 2600, 2.30, None, None, 0.84),
    ("water", 1000, 0.60, None, None, 4.186),
    ("glass", 2500, 0.96, None, None, 0.84),
    ("gypsum board", 800, 0.16, None, None, 1.09),
    ("plaster", 1600, 0.70, None, None, 1.00),
    ("softwood", 510, 0.12, None, None, 1.38),
    ("hardwood", 720, 0.16, None, None, 1.26),
    ("CLT", 480, 0.13, None, None, 1.60),
    ("mineral wool", 30, 0.040, None, None, 0.84),
    ("EPS", 20, 0.035, None, None, 1.45),
    ("XPS", 35, 0.030, None, None, 1.45),
    ("straw bale", 110, 0.060, None, None, 2.00),
    ("steel", 7850, 50.0, None, None, 0.45),
    ("soil (moist)", 1800, 1.50, None, None, 1.30),
    ("air", 1.2, 0.026, None, None, 1.005),
]

# SI -> IP factors
K_IP = 0.577789        # W/m-K -> Btu/hr-ft-F
RHO_IP = 0.0624280     # kg/m3 -> lb/ft3
CP_IP = 1 / 4.1868     # kJ/kg-K -> Btu/lb-F
R_IP = 5.678263        # m2-K/W -> hr-ft2-F/Btu
M_PER_FT = 0.3048
UNITS = {
    "SI": dict(L="m", A="m²", k="W/m·K", rho="kg/m³", cp="J/kg·K", R="m²·K/W", U="W/m²·K",
               UA="W/K", C="J/K", Ca="kJ/m²·K"),
    "IP": dict(L="ft", A="ft²", k="Btu/hr·ft·°F", rho="lb/ft³", cp="Btu/lb·°F", R="hr·ft²·°F/Btu",
               U="Btu/hr·ft²·°F", UA="Btu/hr·°F", C="Btu/°F", Ca="Btu/ft²·°F"),
}
DAY = 86400.0


def generic_table():
    df = pd.DataFrame(GENERIC, columns=TEMPLATE_COLUMNS[:-1])
    df["source"] = "generic typical value"
    return df


def _clean(df):
    """Normalise a materials table: names, numbers, k from R where needed."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    missing = [c for c in ("material", "density_kg_m3", "cp_kJ_kgK") if c not in df]
    if missing:
        raise ValueError(f"materials table is missing column(s) {missing}; "
                         f"expected {', '.join(TEMPLATE_COLUMNS)}")
    for c in TEMPLATE_COLUMNS:
        if c not in df:
            df[c] = np.nan if c != "source" else ""
    df["material"] = df["material"].astype(str).str.strip()
    df = df[df["material"].ne("") & df["material"].ne("nan")]
    for c in ("density_kg_m3", "k_W_mK", "R_m2K_W", "R_thickness_mm", "cp_kJ_kgK"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    from_R = df["k_W_mK"].isna() & df["R_m2K_W"].gt(0) & df["R_thickness_mm"].gt(0)
    df.loc[from_R, "k_W_mK"] = df.loc[from_R, "R_thickness_mm"] / 1000 / df.loc[from_R, "R_m2K_W"]
    df["k_from_R"] = from_R
    df["source"] = df["source"].fillna("").astype(str)
    return df[TEMPLATE_COLUMNS + ["k_from_R"]].reset_index(drop=True)


def read_table(source):
    """Read a materials CSV from a path, URL, bytes or text."""
    if isinstance(source, bytes):
        source = source.decode("utf-8-sig")
    if isinstance(source, str) and source.startswith(("http://", "https://")):
        if "docs.google.com/spreadsheets" in source and "/edit" in source:   # Google Sheet link
            source = source.split("/edit")[0] + "/export?format=csv"
        with urllib.request.urlopen(source, timeout=60) as r:
            source = r.read().decode("utf-8-sig")
    if isinstance(source, str) and not os.path.exists(source):
        return _clean(pd.read_csv(io.StringIO(source)))
    return _clean(pd.read_csv(source, encoding="utf-8-sig"))


def read_google_sheet(name="materials"):
    """Read a Google Sheet from the user's Drive by its name (Colab only; asks once for
    permission). Opening materials.csv with Google Sheets makes a Sheet called 'materials'
    or 'materials.csv'; both are tried."""
    try:
        from google.colab import auth  # type: ignore
    except ImportError:
        raise RuntimeError("Reading a Google Sheet by name needs Colab. Locally, download the "
                           "Sheet as CSV and use read_table('file.csv').") from None
    import gspread  # preinstalled in Colab
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    name = str(name).strip()
    stem = name[:-4] if name.lower().endswith(".csv") else name
    for title in dict.fromkeys([name, stem, stem + ".csv"]):
        try:
            sh = gc.open(title)
            break
        except gspread.SpreadsheetNotFound:
            continue
    else:
        similar = [f["name"] for f in gc.list_spreadsheet_files() if stem.lower() in f["name"].lower()]
        raise FileNotFoundError(f"No Google Sheet called '{stem}' in your Drive." +
                                (f" Sheets with similar names: {', '.join(similar)}" if similar else ""))
    rows = sh.sheet1.get_all_values()
    if not rows:
        raise ValueError(f"The Google Sheet '{sh.title}' is empty.")
    print(f"read Google Sheet '{sh.title}' ({len(rows) - 1} rows)")
    return _clean(pd.DataFrame(rows[1:], columns=rows[0]))


class Materials:
    """A materials table: generic values with the user's rows added (user rows win),
    or the user's rows alone (include_generic=False)."""

    def __init__(self, user=None, include_generic=True):
        tables = [_clean(generic_table())] if include_generic or user is None else []
        if user is not None:
            tables.append(user if "k_from_R" in user else _clean(user))
        df = pd.concat(tables, ignore_index=True)
        df["key"] = df["material"].str.lower()
        self.df = df.drop_duplicates("key", keep="last").drop(columns="key").reset_index(drop=True)
        self.n_user = 0 if user is None else len(user)

    def names(self):
        return list(self.df["material"])

    def find(self, name):
        """Row for a material name (case-insensitive; suggests close matches)."""
        key = str(name).strip().lower()
        hit = self.df[self.df["material"].str.lower() == key]
        if len(hit):
            return hit.iloc[0]
        part = self.df[self.df["material"].str.lower().str.contains(key, regex=False)]
        if len(part) == 1:
            return part.iloc[0]
        close = list(part["material"]) or difflib.get_close_matches(name, self.names(), n=8, cutoff=0.4)
        raise KeyError(f"'{name}' is not in the materials table." +
                       (f" Did you mean: {'; '.join(close[:12])}?" if close else ""))

    def search(self, text=""):
        t = str(text).strip().lower()
        df = self.df if not t else self.df[self.df["material"].str.lower().str.contains(t, regex=False)]
        return df

    def show(self, text="", units="SI"):
        df = self.search(text)
        out = pd.DataFrame({"material": df["material"]})
        k, rho, cp = props_in(df["k_W_mK"], df["density_kg_m3"], df["cp_kJ_kgK"], units)
        u = UNITS[units]
        out[f"density [{u['rho']}]"] = rho
        out[f"k [{u['k']}]"] = k
        out[f"c [{u['cp']}]"] = cp
        out["note"] = np.where(df["k_from_R"], "k from R at " + df["R_thickness_mm"].astype(str) + " mm", "")
        out["source"] = df["source"]
        return out.reset_index(drop=True)


# The material picked in the browser (step 2); step 3 uses it when its own box is blank.
SELECTED = {"material": "concrete"}


def browse(mats, units="SI", whole_table=True):
    """Show the whole materials table and a list to pick a material from.
    The pick is stored in SELECTED['material'] and its properties are shown."""
    import ipywidgets as W
    from IPython.display import display, clear_output
    if whole_table:
        table = mats.show("", units).round(4)
        try:   # Colab: sortable, searchable table
            from google.colab import data_table  # type: ignore
            display(data_table.DataTable(table, include_index=False, num_rows_per_page=15))
        except ImportError:
            with pd.option_context("display.max_rows", None):
                display(table)
    u = UNITS[units]
    filt = W.Text(placeholder="type part of a name to narrow the list", description="Filter:",
                  layout=W.Layout(width="420px"))
    pick = W.Select(rows=14, layout=W.Layout(width="420px"))
    out = W.Output(layout=W.Layout(padding="0 0 0 16px"))

    def refresh(*_):
        df = mats.search(filt.value)
        current = pick.value
        pick.options = [(r.material, r.material) for r in df.itertuples()]
        if current in list(df["material"]):
            pick.value = current

    def show(change=None):
        name = pick.value
        with out:
            clear_output()
            if not name:
                print("No material selected.")
                return
            SELECTED["material"] = name
            row = mats.find(name)
            k, rho, cp = props_in(row.k_W_mK, row.density_kg_m3, row.cp_kJ_kgK, units)
            print(f"selected: {name}\n")
            print(f"  density   {rho:,.4g} {u['rho']}")
            print(f"  k         {k:,.4g} {u['k']}" + (f"   (from R = {row.R_m2K_W:g} m²·K/W at "
                                                     f"{row.R_thickness_mm:g} mm)" if row.k_from_R else ""))
            print(f"  c         {cp:,.4g} {u['cp']}")
            if not pd.isna(row.k_W_mK) and not pd.isna(row.density_kg_m3) and not pd.isna(row.cp_kJ_kgK):
                d = penetration_depth(row.k_W_mK, row.density_kg_m3, row.cp_kJ_kgK)
                print(f"  daily penetration depth δ  {d if units == 'SI' else d / M_PER_FT:.3g} {u['L']}")
            if row.source:
                print(f"  source    {row.source}")
            print("\nRun step 3 to get layer values for it.")

    filt.observe(refresh, names="value")
    pick.observe(show, names="value")
    refresh()
    if SELECTED["material"] in pick.options or any(v == SELECTED["material"] for _, v in pick.options):
        pick.value = SELECTED["material"]
    show()
    display(W.VBox([filt, W.HBox([pick, out])]))
    return pick


def props_in(k_si, rho_si, cp_kj, units):
    """(k, rho, cp) in the network's units; cp in J/kg-K for SI (as the model's formulas use)."""
    if units == "SI":
        return k_si, rho_si, cp_kj * 1000.0
    return k_si * K_IP, rho_si * RHO_IP, cp_kj * CP_IP


def penetration_depth(k_si, rho_si, cp_kj, period_s=DAY):
    """Depth (m) at which a periodic temperature swing falls to 1/e: sqrt(alpha*P/pi)."""
    alpha = k_si / (rho_si * cp_kj * 1000.0)
    return math.sqrt(alpha * period_s / math.pi)


def _g(x, n=5):
    """Short number for formulas."""
    return f"{float(x):.{n}g}"


def layer(mats, material, area, thickness, units="SI", n_segments=0):
    """Values for one layer. area, thickness in m/m² (SI) or ft/ft² (IP).
    n_segments = 0 -> suggested from the daily penetration depth."""
    row = mats.find(material)
    if pd.isna(row.k_W_mK) or pd.isna(row.density_kg_m3) or pd.isna(row.cp_kJ_kgK):
        raise ValueError(f"'{row.material}' needs density, specific heat and k (or R and thickness).")
    k, rho, cp = props_in(row.k_W_mK, row.density_kg_m3, row.cp_kJ_kgK, units)
    L_m = thickness if units == "SI" else thickness * M_PER_FT
    delta = penetration_depth(row.k_W_mK, row.density_kg_m3, row.cp_kJ_kgK)
    # segments no thicker than half of delta; light layers (< 20 kJ/m2K) need only one
    light = row.density_kg_m3 * row.cp_kJ_kgK * L_m < 20.0
    suggested = 1 if light else max(1, math.ceil(2 * L_m / delta))
    n = int(n_segments) if n_segments and n_segments > 0 else suggested
    seg = thickness / n
    u = UNITS[units]
    R = thickness / k
    C = rho * cp * area * thickness
    rows = [
        ("thermal resistance R = L/k", R, u["R"]),
        ("U = k/L (no surface films)", 1 / R, u["U"]),
        ("whole layer: conduction UA = k·A/L", k * area / thickness, u["UA"]),
        ("whole layer: capacitance C = ρ·c·A·L", C, u["C"]),
        ("heat capacity per area ρ·c·L", rho * cp * thickness / (1000.0 if units == "SI" else 1.0), u["Ca"]),
        (f"per segment (L/{n}): conduction", k * area / seg, u["UA"]),
        ("per segment: capacitance", rho * cp * area * seg, u["C"]),
        ("surface → first segment centre (L/2n): conduction", k * area / (seg / 2), u["UA"]),
        ("daily penetration depth δ", delta if units == "SI" else delta / M_PER_FT, u["L"]),
    ]
    table = pd.DataFrame([(q, f"{v:,.4g}", un) for q, v, un in rows], columns=["quantity", "value", "units"])
    kf = f"k={_g(k)}"
    formulas = {
        "per segment conduction": f"conduction(A={_g(area)}, L={_g(seg)}, {kf})",
        "per segment capacitance": f"mass({_g(area)}, {_g(seg)}, rho={_g(rho)}, cp={_g(cp)})",
        "surface → first segment": f"conduction(A={_g(area)}, L={_g(seg / 2)}, {kf})",
        "whole layer conduction": f"conduction(A={_g(area)}, L={_g(thickness)}, {kf})",
        "whole layer capacitance": f"mass({_g(area)}, {_g(thickness)}, rho={_g(rho)}, cp={_g(cp)})",
    }
    info = dict(material=row.material, k=k, rho=rho, cp=cp, n=n, suggested=suggested,
                k_from_R=bool(row.k_from_R), source=row.source)
    return table, formulas, info


def assembly(mats, layers_text, area=1.0, units="SI", R_inside=None, R_outside=None):
    """Layers listed one per line as 'material, thickness' (inside to outside).
    Returns a per-layer table and totals. Surface resistances default to
    0.12 (inside) and 0.03 (outside) m²K/W, converted for IP."""
    R_in = (0.12 if units == "SI" else 0.12 * R_IP) if R_inside is None else R_inside
    R_out = (0.03 if units == "SI" else 0.03 * R_IP) if R_outside is None else R_outside
    rows = []
    for line in str(layers_text).splitlines():
        line = line.split("#")[0].strip()
        if not line:
            continue
        name, _, t = line.rpartition(",")
        if not name:
            raise ValueError(f"write each layer as 'material, thickness' (got '{line}')")
        t = float(t)
        _, _, info = layer(mats, name.strip(), area, t, units)
        R = t / info["k"]
        Ca = info["rho"] * info["cp"] * t
        rows.append(dict(material=info["material"], thickness=t, k=info["k"], R=R,
                         heat_capacity_per_area=Ca, suggested_segments=info["suggested"]))
    u = UNITS[units]
    df = pd.DataFrame(rows)
    R_total = R_in + R_out + (df["R"].sum() if len(df) else 0.0)
    totals = dict(R_surfaces=R_in + R_out, R_total=R_total, U=1 / R_total, UA=area / R_total,
                  C=(df["heat_capacity_per_area"].sum() * area) if len(df) else 0.0)
    df = df.rename(columns={"thickness": f"thickness [{u['L']}]", "k": f"k [{u['k']}]",
                            "R": f"R [{u['R']}]",
                            "heat_capacity_per_area": f"ρ·c·L [{u['C']} per {u['A']}]"})
    return df, totals

from IPython.display import display, Markdown
try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def drive_path(folder="ThermalMatrixModel"):
    """Folder in the student's Google Drive (mounted on first use); a local folder outside Colab."""
    if IN_COLAB:
        if not os.path.isdir("/content/drive/MyDrive"):
            from google.colab import drive
            drive.mount("/content/drive")
        path = os.path.join("/content/drive/MyDrive", folder.strip("/ ") or "ThermalMatrixModel")
    else:
        path = folder.strip("/ ") or "ThermalMatrixModel"
        print(f"(not in Colab: using the local folder '{path}' instead of Google Drive)")
    os.makedirs(path, exist_ok=True)
    return path
print(f'✓ property calculator loaded - version {CALC_VERSION}')

## 1 ▸ Materials table

In [ ]:
#@title 1 ▸ Load the materials table { display-mode: "form" }
#@markdown **My Google Sheet**: a Google Sheet in your Drive called `sheet_name` (e.g. the one made by opening the template with Google Sheets).
#@markdown **My CSV in Google Drive**: the file `csv_name` in *My Drive ▸ drive_folder*. **Link**: a Google Sheet shared as "anyone with the link", or a CSV URL.
materials_from = "My Google Sheet" #@param ["My Google Sheet", "Generic values only", "My CSV in Google Drive", "Upload a CSV", "Google Sheet or CSV link"]
sheet_name = "materials" #@param {type:"string"}
drive_folder = "ThermalMatrixModel" #@param {type:"string"}
csv_name = "materials.csv" #@param {type:"string"}
link = "" #@param {type:"string"}
#@markdown Untick to use only your table (once it is complete); ticked, the generic values fill in anything your table lacks.
include_generic = True #@param {type:"boolean"}
#@markdown Tick to save a template (generic rows + the right columns) as `csv_name` in your Drive folder; open it with Google Sheets to edit.
save_template = False #@param {type:"boolean"}

if save_template:
    path = os.path.join(drive_path(drive_folder), csv_name)
    if os.path.exists(path):
        print(f"{path} already exists - not overwritten (pick another csv_name)")
    else:
        generic_table()[TEMPLATE_COLUMNS].to_csv(path, index=False)
        print("template saved to", path, "- in Google Drive open it with Google Sheets, add rows, "
              "then load it here with 'My Google Sheet'")

user = None
if materials_from.startswith("My Google Sheet"):
    try:
        user = read_google_sheet(sheet_name)
    except (FileNotFoundError, RuntimeError) as e:
        print(e)
        print("-> using the generic values (tick save_template to start your own table)")
elif materials_from.startswith("My CSV"):
    user = read_table(os.path.join(drive_path(drive_folder), csv_name))
elif materials_from.startswith("Upload"):
    if not IN_COLAB:
        raise RuntimeError("Uploading needs Colab. Locally use: user = read_table('file.csv')")
    up = _colab_files.upload()
    user = read_table(next(iter(up.values())))
elif materials_from.startswith("Google Sheet"):
    user = read_table(link)
mats = Materials(user, include_generic)
print(f"{len(mats.df)} materials" + (f" ({mats.n_user} rows read from your table, added to the generic values)" if mats.n_user else " (generic values)"))

In [ ]:
#@title 2 ▸ Browse and select a material { display-mode: "form" }
#@markdown The whole table (click a column heading to sort, or use its search box), then a list to **pick a material** from:
#@markdown click one to see its properties; step 3 uses it. Type in *Filter* to narrow the list.
units = "SI" #@param ["SI", "IP"]
show_whole_table = True #@param {type:"boolean"}
browse(mats, units, whole_table=show_whole_table);

## 2 ▸ One layer
Geometry in the network's units: m and m² for SI, ft and ft² for IP. **area** = face area the
heat passes through; **thickness** = layer thickness in the direction of heat flow.

**How many segments?** A massive layer is modelled as several capacitive nodes (like the
spreadsheet's *Wall 1–5*). A daily temperature swing dies away over the **penetration depth**
δ = √(α·P/π) (α = k/ρc, P = 1 day); segments no thicker than δ/2 follow it well, so that is the
suggestion when `n_segments` is 0. Light or insulating layers need only one.

In [ ]:
#@title 3 ▸ Layer values { display-mode: "form" }
#@markdown `material`: blank = the one selected in step 2 (or type a name).
material = "" #@param {type:"string"}
units = "SI" #@param ["SI", "IP"]
area = 48 #@param {type:"number"}
thickness = 0.4 #@param {type:"number"}
n_segments = 0 #@param {type:"integer"}

table, formulas, info = layer(mats, material.strip() or SELECTED["material"], area, thickness, units, n_segments)
u = UNITS[units]
print(f"{info['material']}: k = {info['k']:.4g} {u['k']}, ρ = {info['rho']:.4g} {u['rho']}, "
      f"c = {info['cp']:.4g} {u['cp']}" + ("  (k from R)" if info["k_from_R"] else "") +
      (f"  [{info['source']}]" if info["source"] else ""))
print(f"{info['n']} segment(s)" + ("" if info["n"] == info["suggested"] else f"  (suggested: {info['suggested']})"))
display(table.round(3))
display(Markdown("**Formulas to paste into the network editor** (" + units + " network):\n\n" +
                 "\n".join(f"- {k}: `{v}`" for k, v in formulas.items())))

## 3 ▸ An assembly of layers
List the layers from **inside to outside**, one per line as `material, thickness`. The table
gives each layer's R and heat capacity, and the totals with surface films (defaults: inside
0.12, outside 0.03 m²·K/W; set to change them). Use it for the steady-state U-value of a wall
and to see where its mass sits.

In [ ]:
#@title 4 ▸ Assembly { display-mode: "form" }
units = "SI" #@param ["SI", "IP"]
area = 1 #@param {type:"number"}
R_inside = 0 #@param {type:"number"}
R_outside = 0 #@param {type:"number"}
#@markdown Edit the layers in the code (*Show code*, or double-click this cell):
layers = """
gypsum board, 0.0125
mineral wool, 0.15
brick, 0.10
"""
df, tot = assembly(mats, layers, area, units, R_inside or None, R_outside or None)
u = UNITS[units]
display(df.round(4))
print(f"R total (with surface films {tot['R_surfaces']:.3g}) = {tot['R_total']:.3g} {u['R']}   "
      f"U = {tot['U']:.3g} {u['U']}   UA = {tot['UA']:.4g} {u['UA']}   C = {tot['C']:.4g} {u['C']} for {area:g} {u['A']}")

### Version history
- **2026-11-24 19:25** - [property calculator] Step 2 shows the whole materials table and a list to pick a material from (step 3 uses the pick); your Google Sheet 'materials' is the default table, with an option to leave out the generic values
- **2026-11-24 19:15** - [property calculator] Read the materials table straight from a Google Sheet in your Drive by name (no need to save as CSV); template instructions updated
- **2026-11-24 18:53** - [property calculator] First version: materials table (generic values plus your own CSV or Google Sheet in the layout of ASHRAE Fundamentals ch. 26 Table 1), layer values with suggested segments and paste-ready formulas, and assembly U-values